In [71]:
import sys
import math
import logging

import numpy as np
import scipy.stats as stats
from tabulate import tabulate

In [2]:
sys.path.append("../")

In [3]:
import plot_utils

In [4]:
logging.getLogger().setLevel(logging.ERROR)

In [9]:
plots_CR = plot_utils.loader(tag="full_analysis_Apr2025_CR", input_dir="../../")
plots_SR = plot_utils.loader(tag="full_analysis_Apr2025_SRs", input_dir="../../")
plots = {}
for dataset in plots_CR:
    plots[dataset] = plots_CR[dataset] | plots_SR[dataset]

In [11]:
slice_hists = {
    "SR_low_temp_loose": slice(4j, None),
    "SR_low_temp_tight": slice(3j, None),
    "SR_high_temp_loose": slice(4j, None),
    "SR_high_temp_tight": slice(3j, None),
}
qcd_extrapolation = plot_utils.Extrapolation(
    plots["QCD_Pt_MuEnrichedPt5_2018"], uncertainty_scheme="full"
)
qcd_extrapolation.extrapolate(slice_hists=slice_hists, verbose=False)
slice_hists = {
    "SR_low_temp_loose": slice(4j, None),
    "SR_low_temp_tight": slice(3j, None),
    "SR_high_temp_loose": slice(4j, None),
    "SR_high_temp_tight": slice(3j, None),
}
dy_extrapolation = plot_utils.Extrapolation(plots["DY_2018"], uncertainty_scheme="full")
dy_extrapolation.extrapolate(slice_hists=slice_hists, verbose=False)

/uscms_data/d3/chpapage/SUEPs/MuonTriggers/muon_branches/SUEPCoffea_dask/plotting/notebooks/../plot_utils.py:627: RuntimeWarning: invalid value encountered in divide
  np.sqrt(mc_vars) / mc_vals,


In [22]:
histogram = plots["TT_powheg_2018"]["SR_high_temp_tight"]

In [24]:
histogram[::sum].variance / histogram[::sum].value

0.03775742523795109

In [30]:
def get_poisson_errors(N, alpha=0.6827, scale=1):
    # Return the Garwood confidence interval for a Poisson distribution
    upper = stats.gamma.ppf((1 + alpha) / 2, N + 1) - N
    lower = N - stats.gamma.ppf((1 - alpha) / 2, N)
    return np.nan_to_num(lower) * scale, np.nan_to_num(upper) * scale

In [32]:
get_poisson_errors(0, scale=0.1)

(0.0, 0.18410547609529326)

In [137]:
mc_procs = [
    ("Higgs_2018", "Higgs"),
    ("TTV_2018", "TTV"),
    ("ST_NLO_2018", "ST"),
    ("WJets_2018", "WJets"),
    ("VV+VVV_2018", "VV+VVV"),
    ("TT_powheg_2018", "TT"),
    ("DY_2018", "DY"),
    ("QCD_Pt_MuEnrichedPt5_2018", "QCD"),
]

In [138]:
def smart_round(x):
    if x == 0:
        return x
    if x > 10:
        return round(x)
    return round(x, -math.floor(np.log10(x)) + 1)

In [141]:
regions = ["SR_high_temp_tight", "SR_low_temp_tight"]

for region in regions:
    header = [
        "process",
        "highest populated nMuon bin",
        "scale (sumw2/sumw)",
        "MC events ((sumw)^2/sumw2)",
        "Poisson unc. band for y=0",
    ]
    table = []
    for proc, label in mc_procs:
        h = plots[proc][region]
        # Get scale of rightmost non-zero bin
        values = h.values()[h.values() > 0]
        variances = h.variances()[h.values() > 0]
        scale = variances[-1] / values[-1]
        mc_events = values[-1] ** 2 / variances[-1]
        unc_band = get_poisson_errors(0, scale=scale)
        row = [
            label,
            np.where(plots[proc][region].values() > 0)[0][-1] + 3,
            smart_round(scale),
            round(mc_events, 1),
            (smart_round(unc_band[0]), smart_round(unc_band[1])),
        ]
        table.append(row)

    print(region)
    print()
    print(tabulate(table, headers=header))
    print(112 * "-")
    print()

SR_high_temp_tight

process      highest populated nMuon bin    scale (sumw2/sumw)    MC events ((sumw)^2/sumw2)  Poisson unc. band for y=0
---------  -----------------------------  --------------------  ----------------------------  ---------------------------
Higgs                                  6               0.00041                           3.7  (0.0, 0.00075)
TTV                                    5               0.0048                            1.1  (0.0, 0.0089)
ST                                     5               0.053                             1    (0.0, 0.097)
WJets                                  4               0.65                              4.8  (0.0, 1.2)
VV+VVV                                 6               0.00078                          37.9  (0.0, 0.0014)
TT                                     6               0.037                             1    (0.0, 0.067)
DY                                     5               0.82                              4    

In [111]:
smart_round(133.132423)

'130.0'

In [119]:
round(128432.23234234, -4)

130000.0